In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import numpy as np
from sklearn.model_selection import train_test_split


In [2]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure your config file is imported correctly so the script knows where your data is
from config import WINDOWS_DIR

ModuleNotFoundError: No module named 'seaborn'

In [3]:
class EEGDataset(Dataset):
    def __init__(self, npz_files):
        self.X = []
        self.y = []
        for f in npz_files:
            data = np.load(f)
            self.X.append(data['X'])
            self.y.append(data['y'])
            
        self.X = np.concatenate(self.X, axis=0)
        self.y = np.concatenate(self.y, axis=0)
        
        # Apply Z-score normalization per window and channel
        mean = np.mean(self.X, axis=(-2, -1), keepdims=True)
        std = np.std(self.X, axis=(-2, -1), keepdims=True) + 1e-8
        self.X = (self.X - mean) / std

    def __len__(self):
        return len(self.y)
        
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.long)


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [4]:
def evaluate_model(model, test_loader, device, patient_id):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    print("\n" + "="*50)
    print("--- Classification Report ---")
    target_names = ["Interictal (0)", "Preictal (1)", "Ictal (2)"]
    print(classification_report(all_labels, all_preds, target_names=target_names))
    
    # Generate Heatmap
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=["Normal (0)", "Warning (1)", "Seizure (2)"],
                yticklabels=["Normal (0)", "Warning (1)", "Seizure (2)"])
    
    plt.title(f'AI Predictions vs Actual States (Patient: {patient_id})')
    plt.ylabel('Actual True State')
    plt.xlabel('AI Predicted State')
    
    os.makedirs("plots", exist_ok=True)
    plot_path = f"plots/confusion_matrix_{patient_id}.png"
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"\n--> Visual Confusion Matrix saved to: {plot_path}")
    print("="*50 + "\n")

In [9]:
class EEGSpectrogramCNNLSTM(nn.Module):
    # CHANGED: in_channels is now exactly 23 to match your data
    def __init__(self, num_classes=3, in_channels=23): 
        super(EEGSpectrogramCNNLSTM, self).__init__()
        
        # 1. The Visual Feature Extractor (CNN)
        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveMaxPool2d((8, 8)) 
        )
        
        # 2. The Timeline Processor (LSTM)
        self.lstm = nn.LSTM(
            input_size=512, 
            hidden_size=128, 
            num_layers=2, 
            batch_first=True, 
            dropout=0.3
        )
        
        # 3. The Final Decision (Classifier)
        self.fc1 = nn.Linear(128, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        c = self.cnn(x) 
        batch_size, channels, freq, time = c.size()
        
        # Reshape for LSTM: (Batch, Sequence/Time, Features)
        c = c.permute(0, 3, 1, 2) 
        c = c.reshape(batch_size, time, channels * freq) 
        
        lstm_out, _ = self.lstm(c)
        last_time_step = lstm_out[:, -1, :] 
        
        out = self.fc1(last_time_step)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out

In [10]:
import os
import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split

# This line fixes your exact error:
from config import WINDOWS_DIR

def train_lopo(max_epochs=20, batch_size=64, initial_lr=0.0003, patience=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on device: {device}")
    
    all_npz_files = list(WINDOWS_DIR.glob("*_spectrograms.npz"))
    all_patients = sorted(list(set([f.name.split('_')[0] for f in all_npz_files])))
    
    os.makedirs("models", exist_ok=True)
    
    for test_patient in all_patients:
        print(f"\n{'='*60}")
        print(f"Fold: Testing on {test_patient} (Training on all others)")
        print(f"{'='*60}")
        
        train_pool_files = [f for f in all_npz_files if test_patient not in f.name]
        test_files = [f for f in all_npz_files if test_patient in f.name]
        
        if not test_files:
            continue
            
        train_files, val_files = train_test_split(train_pool_files, test_size=0.2, random_state=42)
            
        print("Loading training and validation data into memory...")
        train_dataset = EEGDataset(train_files)
        val_dataset = EEGDataset(val_files)
        
        # Balanced Sampler
        class_counts = np.bincount(train_dataset.y, minlength=3)
        class_counts = np.where(class_counts == 0, 1, class_counts)
        class_weights = 1.0 / class_counts
        sample_weights = class_weights[train_dataset.y]
        
        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        
        print(f"Class distribution in train split: {class_counts}")
        
        # Load the new LSTM architecture
        model = EEGSpectrogramCNNLSTM(num_classes=3).to(device)
        criterion = FocalLoss(gamma=2.0)
        optimizer = optim.Adam(model.parameters(), lr=initial_lr)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
        
        best_val_loss = float('inf')
        epochs_no_improve = 0
        model_path = f"models/lstm_fold_{test_patient}.pth"
        
        for epoch in range(max_epochs):
            # Train Phase
            model.train()
            running_train_loss = 0.0
            
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                running_train_loss += loss.item()
                
            avg_train_loss = running_train_loss / len(train_loader)
            
            # Validation Phase
            model.eval()
            running_val_loss = 0.0
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    running_val_loss += loss.item()
                    
            avg_val_loss = running_val_loss / len(val_loader)
            
            print(f"Epoch [{epoch+1}/{max_epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
            
            scheduler.step(avg_val_loss)
            
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                epochs_no_improve = 0
                torch.save(model.state_dict(), model_path)
                print(f"  [*] Val loss improved. Saved checkpoint.")
            else:
                epochs_no_improve += 1
                print(f"  [!] No improvement for {epochs_no_improve} epoch(s).")
                
            if epochs_no_improve >= patience:
                print(f"\nEarly stopping triggered at epoch {epoch+1}.")
                break
                
        print(f"\nFinished fold training. Best model saved to {model_path}")
        
        # Test Phase
        print(f"Loading test data for {test_patient}...")
        test_dataset = EEGDataset(test_files)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False) 
        
        # Load the best weights before evaluating
        model.load_state_dict(torch.load(model_path, weights_only=True, map_location=device))
        evaluate_model(model, test_loader, device, test_patient)
        
        break # We break here to only test on the first fold for now.

# Start the training!
if __name__ == "__main__":
    train_lopo()

Training on device: cuda

Fold: Testing on chb01 (Training on all others)
Loading training and validation data into memory...


MemoryError: Unable to allocate 16.2 GiB for an array with shape (161453, 23, 65, 9) and data type float64